## MODELOS ENTRENADOS

In [8]:
from scipy.ndimage import binary_erosion
from scipy.spatial import cKDTree
from scipy.spatial.distance import directed_hausdorff
import numpy as np

def get_boundary(mask, width=2):
    eroded = binary_erosion(mask, iterations=width)
    return mask & ~eroded

def boundary_iou(pred_mask, gt_mask, width=2):
    pred_boundary = get_boundary(pred_mask, width)
    gt_boundary   = get_boundary(gt_mask, width)
    intersection  = np.logical_and(pred_boundary, gt_boundary).sum()
    union         = np.logical_or(pred_boundary, gt_boundary).sum()
    return intersection / union if union > 0 else 0

# TARDA MUCHO PORQUE TIENE QUE COGER TODOS LOS PUNTOS DE LA MÁSCARA
"""def assd(pred_mask, gt_mask):
    pred_points = np.argwhere(pred_mask)
    gt_points   = np.argwhere(gt_mask)
    
    if len(pred_points) == 0 or len(gt_points) == 0:
        return 0
    
    d_pred_to_gt = cKDTree(gt_points).query(pred_points)[0].mean()
    d_gt_to_pred = cKDTree(pred_points).query(gt_points)[0].mean()
    
    return (d_pred_to_gt + d_gt_to_pred) / 2"""

def hausdorff_95(pred_mask, gt_mask):
    pred_points = np.argwhere(pred_mask)
    gt_points   = np.argwhere(gt_mask)
    if len(pred_points) == 0 or len(gt_points) == 0:
        return 0
    d1 = directed_hausdorff(pred_points, gt_points)[0]
    d2 = directed_hausdorff(gt_points, pred_points)[0]
    return np.percentile([d1, d2], 95)


In [9]:
import time
import torch

def measure_inference(predictor, image, point_coords, point_labels):
    if torch.cuda.is_available():
        vram_before = torch.cuda.memory_allocated() / 1024**2
    
    start   = time.time()
    predictor.set_image(image)
    masks, scores, _ = predictor.predict(point_coords=point_coords, point_labels=point_labels)
    latency = (time.time() - start) * 1000  # Está en ms

    if torch.cuda.is_available():
        vram = torch.cuda.memory_allocated() / 1024**2 - vram_before
    else:
        vram = 0

    return masks, scores, latency, vram

def measure_inference_sam3_prompt(predictor, img_path, text_prompt):
    if torch.cuda.is_available():
        vram_before = torch.cuda.memory_allocated() / 1024**2

    start = time.time()
    predictor.set_image(img_path)
    predictor.model.set_classes(text=["polyp"])
    predictor.prompts["text"] = [text_prompt]
    results = predictor()
    
    latency = (time.time() - start) * 1000

    if torch.cuda.is_available():
        vram = torch.cuda.memory_allocated() / 1024**2 - vram_before
    else:
        vram = 0

    return results, latency, vram

In [10]:
import numpy as np

def compute_all_metrics(pred_mask, gt_mask):
    """a"""
    tp = np.logical_and(pred_mask, gt_mask).sum()
    fp = np.logical_and(pred_mask, ~gt_mask).sum()
    fn = np.logical_and(~pred_mask, gt_mask).sum()
    tn = np.logical_and(~pred_mask, ~gt_mask).sum()

    iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
    dice = (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f2 = 5 * (precision * recall) / (4 * precision + recall) if (4 * precision + recall) > 0 else 0
    pixel_accuracy = (tp + tn) / (tp + fp + fn + tn) if (tp + fp + fn + tn) > 0 else 0
    return iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy

In [11]:
import os
import shutil
import random

dataset = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\Kvasir-SEG"

def split_dataset(dataset, train_ratio=0.7, val_ratio=0.15):
    images_dir = os.path.join(dataset, "images")
    masks_dir = os.path.join(dataset, "masks")

    images = [f.replace(".jpg", "") for f in os.listdir(images_dir) if f.endswith(".jpg")]
    random.seed(42)
    random.shuffle(images)

    n = len(images)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    splits = {
        "train": images[:n_train],
        "val": images[n_train:n_train + n_val],
        "test": images[n_train+n_val:]
    }

    for split, names in splits.items():
        os.makedirs(os.path.join(dataset, "images", split), exist_ok=True)
        os.makedirs(os.path.join(dataset, "masks",  split), exist_ok=True)
        for name in names:
            shutil.copy(os.path.join(images_dir, name + ".jpg"), os.path.join(dataset, "images", split, name + ".jpg"))
            shutil.copy(os.path.join(masks_dir,  name + ".jpg"), os.path.join(dataset, "masks",  split, name + ".jpg"))

split_dataset(dataset)


In [12]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from segment_anything import sam_model_registry
import cv2
import numpy as np
import os
import json

torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"

class SegDataset(Dataset):
    def __init__(self, dataset_path, split, bbox_json, img_size=1024):
        self.img_size   = img_size
        self.images_dir = os.path.join(dataset_path, "images", split)
        self.masks_dir  = os.path.join(dataset_path, "masks",  split)
        self.samples    = []

        with open(bbox_json) as f:
            bbox_json = json.load(f)

        for img_name, info in bbox_json.items():
            img_path  = os.path.join(self.images_dir, img_name + ".jpg")
            mask_path = os.path.join(self.masks_dir,  img_name + ".jpg")
            if os.path.exists(img_path) and os.path.exists(mask_path):
                self.samples.append((img_path, mask_path, info["bbox"][0]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path, bbox = self.samples[idx]

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (self.img_size, self.img_size))
        image = torch.tensor(image).permute(2, 0, 1).float() / 255.0

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (256, 256))
        mask = torch.tensor((mask > 127).astype(np.float32)).unsqueeze(0)

        xmin, ymin, xmax, ymax = bbox["xmin"], bbox["ymin"], bbox["xmax"], bbox["ymax"]
        orig_w = cv2.imread(img_path).shape[1]
        orig_h = cv2.imread(img_path).shape[0]

        scale_x = self.img_size / orig_w
        scale_y = self.img_size / orig_h

        point = torch.tensor([[(xmin + (xmax-xmin)/2) * scale_x, (ymin + (ymax-ymin)/2) * scale_y]]).float()
        label = torch.tensor([1])

        return image, mask, point, label


def train_sam(dataset_path, bbox_json, weights_path, output_path, vit, epochs=50, batch_size=4):
    
    sam = sam_model_registry[vit](checkpoint=weights_path)
    sam.to(device)

    for param in sam.image_encoder.parameters():
        param.requires_grad = False
    for param in sam.prompt_encoder.parameters():
        param.requires_grad = False

    optimizer = torch.optim.Adam(sam.mask_decoder.parameters(), lr=1e-4)
    scheduler  = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
    loss_fn   = nn.BCEWithLogitsLoss()

    train_dataset = SegDataset(dataset_path, "train", bbox_json)
    train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    sam.train()
    for epoch in range(epochs):
        total_loss = 0
        for images, masks, points, labels in train_loader:
            images, masks  = images.to(device), masks.to(device)
            points, labels = points.to(device), labels.to(device)

            loss_batch = 0
            with torch.no_grad():
                image_embeddings = sam.image_encoder(images)

            for i in range(images.shape[0]):
                with torch.no_grad():
                    sparse_embeddings, dense_embeddings = sam.prompt_encoder(
                        points=(points[i].unsqueeze(0), labels[i].unsqueeze(0)),
                        boxes=None,
                        masks=None
                    )

                low_res_masks, _ = sam.mask_decoder(
                    image_embeddings=image_embeddings[i].unsqueeze(0),
                    image_pe=sam.prompt_encoder.get_dense_pe(),
                    sparse_prompt_embeddings=sparse_embeddings,
                    dense_prompt_embeddings=dense_embeddings,
                    multimask_output=False,
                )

                loss_batch += loss_fn(low_res_masks, masks[i].unsqueeze(0))

            loss = loss_batch / images.shape[0]
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        scheduler.step()
        print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")

    torch.save(sam.state_dict(), output_path)
    return output_path


## SAM 1 BASE ENTRENAMIENTO

In [ ]:
import numpy as np
from segment_anything import sam_model_registry, SamPredictor
import os
import cv2
import pandas as pd
import json
import torch

torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"
print(f"Usando GPU: {torch.cuda.get_device_name(0)}")

dataset = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\Kvasir-SEG"
bboxes = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\Kvasir-SEG\\kavsir_bboxes.json"
weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam_b.pt"
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"


def evaluate_model(dataset, weights_path):
    with open(os.path.join(dataset, "kavsir_bboxes.json")) as f:
        bboxes = json.load(f)

    sam = sam_model_registry["vit_b"](checkpoint=weights_path)
    sam.to(device)
    sam.eval()
    predictor = SamPredictor(sam)

    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    dice_scores = []
    specificity_scores = []
    f2_scores = []
    pixel_accuracy_scores = []
    boundary_iou_scores = []
    hausdorff_95_scores = []
    latency_scores = []
    vram_scores = []

    for img_name, info in bboxes.items():
        img_path  = os.path.join(dataset, "images", "test", img_name + ".jpg")
        mask_path = os.path.join(dataset, "masks",  "test", img_name + ".jpg")

        if not os.path.exists(img_path) or not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = np.squeeze(gt_mask)
        gt_mask = (gt_mask > 127).astype(bool)

        bbox_info = info["bbox"][0]

        xmin = bbox_info["xmin"]
        ymin = bbox_info["ymin"]
        xmax = bbox_info["xmax"]
        ymax = bbox_info["ymax"]

        w = xmax - xmin
        h = ymax - ymin

        central_point = [[xmin + w / 2, ymin + h / 2]]

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        masks, scores, latency, vram = measure_inference(predictor, image, np.array(central_point), np.array([1]))

        if masks is None or len(masks) == 0:
            continue
        best_idx  = np.argmax(scores)
        pred_mask = masks[best_idx].astype(bool)

        H, W      = gt_mask.shape
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        f1_scores.append(f1)
        recall_scores.append(recall)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    mean_iou = np.mean(iou_scores)
    mean_f1 = np.mean(f1_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)
    mean_dice = np.mean(dice_scores)
    mean_specificity = np.mean(specificity_scores)
    mean_f2 = np.mean(f2_scores)
    mean_pixel_accuracy = np.mean(pixel_accuracy_scores)
    mean_boundary_iou = np.mean(boundary_iou_scores)
    mean_hausdorff_95 = np.mean(hausdorff_95_scores)
    mean_latency = np.mean(latency_scores)
    mean_vram = np.mean(vram_scores)

    resultados = {
         "modelo": ["sam_b_kvasir"],
         "mean_iou": [mean_iou],
         "mean_f1": [mean_f1],
         "mean_recall": [mean_recall],
         "mean_precision": [mean_precision],
         "mean_dice": [mean_dice],
         "mean_specificity": [mean_specificity],
         "mean_f2": [mean_f2],
         "mean_pixel_accuracy": [mean_pixel_accuracy],
         "mean_boundary_iou": [mean_boundary_iou],
         "mean_hausdorff_95": [mean_hausdorff_95],
         "mean_latency_ms": [mean_latency],
         "mean_vram_mb": [mean_vram]
    }

    df = pd.DataFrame(resultados)
    output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"

    if os.path.exists(output_path):
        df.to_csv(output_path, mode='a', header=False, index=False)
    else:
         df.to_csv(output_path, index=False)

    print(f"Mean IoU: {mean_iou:.4f}")
    print(f"Mean F1 Score: {mean_f1:.4f}")
    print(f"Mean Recall: {mean_recall:.4f}")
    print(f"Mean Precision: {mean_precision:.4f}")
    print(f"Mean Dice: {mean_dice:.4f}")
    print(f"Mean Specificity: {mean_specificity:.4f}")
    print(f"Mean F2: {mean_f2:.4f}")
    print(f"Mean Pixel Accuracy: {mean_pixel_accuracy:.4f}")
    print(f"Mean Boundary IoU: {mean_boundary_iou:.4f}")
    print(f"Mean Hausdorff-95: {mean_hausdorff_95:.4f}")
    print(f"Mean Latency (ms): {mean_latency:.2f}")
    print(f"Mean VRAM Usage (MB): {mean_vram:.2f}")
  
trained_weights = train_sam(dataset, bboxes, weights, output_path, vit="vit_b")
evaluate_model(dataset, trained_weights)


Usando GPU: NVIDIA GeForce RTX 3090
Epoch 1/50 - Loss: 0.1830
Epoch 2/50 - Loss: 0.1351
Epoch 3/50 - Loss: 0.1188
Epoch 4/50 - Loss: 0.1132
Epoch 5/50 - Loss: 0.1065
Epoch 6/50 - Loss: 0.0880
Epoch 7/50 - Loss: 0.0817
Epoch 8/50 - Loss: 0.0995
Epoch 9/50 - Loss: 0.0836
Epoch 10/50 - Loss: 0.0748
Epoch 11/50 - Loss: 0.0685
Epoch 12/50 - Loss: 0.0719


## SAM 1 GRANDE

In [ ]:
import numpy as np
from segment_anything import sam_model_registry, SamPredictor
import os
import cv2
import pandas as pd
import json
import torch

torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"

dataset = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\Kvasir-SEG"
bboxes = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\Kvasir-SEG\\kavsir_bboxes.json"
weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam_l.pt"
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"


def evaluate_model(dataset, weights_path):
    with open(os.path.join(dataset, "kavsir_bboxes.json")) as f:
        bboxes = json.load(f)

    sam = sam_model_registry["vit_l"](checkpoint=weights_path)
    sam.to(device)
    sam.eval()
    predictor = SamPredictor(sam)

    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    dice_scores = []
    specificity_scores = []
    f2_scores = []
    pixel_accuracy_scores = []
    boundary_iou_scores = []
    hausdorff_95_scores = []
    latency_scores = []
    vram_scores = []

    for img_name, info in bboxes.items():
        img_path  = os.path.join(dataset, "images", "test", img_name + ".jpg")
        mask_path = os.path.join(dataset, "masks",  "test", img_name + ".jpg")

        if not os.path.exists(img_path) or not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = np.squeeze(gt_mask)
        gt_mask = (gt_mask > 127).astype(bool)

        bbox_info = info["bbox"][0]

        xmin = bbox_info["xmin"]
        ymin = bbox_info["ymin"]
        xmax = bbox_info["xmax"]
        ymax = bbox_info["ymax"]

        w = xmax - xmin
        h = ymax - ymin

        central_point = [[xmin + w / 2, ymin + h / 2]]

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        masks, scores, latency, vram = measure_inference(predictor, image, np.array(central_point), np.array([1]))

        if masks is None or len(masks) == 0:
            continue
        best_idx  = np.argmax(scores)
        pred_mask = masks[best_idx].astype(bool)

        H, W      = gt_mask.shape
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        f1_scores.append(f1)
        recall_scores.append(recall)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    mean_iou = np.mean(iou_scores)
    mean_f1 = np.mean(f1_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)
    mean_dice = np.mean(dice_scores)
    mean_specificity = np.mean(specificity_scores)
    mean_f2 = np.mean(f2_scores)
    mean_pixel_accuracy = np.mean(pixel_accuracy_scores)
    mean_boundary_iou = np.mean(boundary_iou_scores)
    mean_hausdorff_95 = np.mean(hausdorff_95_scores)
    mean_latency = np.mean(latency_scores)
    mean_vram = np.mean(vram_scores)

    resultados = {
         "modelo": ["sam_l_kvasir"],
         "mean_iou": [mean_iou],
         "mean_f1": [mean_f1],
         "mean_recall": [mean_recall],
         "mean_precision": [mean_precision],
         "mean_dice": [mean_dice],
         "mean_specificity": [mean_specificity],
         "mean_f2": [mean_f2],
         "mean_pixel_accuracy": [mean_pixel_accuracy],
         "mean_boundary_iou": [mean_boundary_iou],
         "mean_hausdorff_95": [mean_hausdorff_95],
         "mean_latency_ms": [mean_latency],
         "mean_vram_mb": [mean_vram]
    }

    df = pd.DataFrame(resultados)
    output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_sam.csv"

    if os.path.exists(output_path):
        df.to_csv(output_path, mode='a', header=False, index=False)
    else:
         df.to_csv(output_path, index=False)

    print(f"Mean IoU: {mean_iou:.4f}")
    print(f"Mean F1 Score: {mean_f1:.4f}")
    print(f"Mean Recall: {mean_recall:.4f}")
    print(f"Mean Precision: {mean_precision:.4f}")
    print(f"Mean Dice: {mean_dice:.4f}")
    print(f"Mean Specificity: {mean_specificity:.4f}")
    print(f"Mean F2: {mean_f2:.4f}")
    print(f"Mean Pixel Accuracy: {mean_pixel_accuracy:.4f}")
    print(f"Mean Boundary IoU: {mean_boundary_iou:.4f}")
    print(f"Mean Hausdorff-95: {mean_hausdorff_95:.4f}")
    print(f"Mean Latency (ms): {mean_latency:.2f}")
    print(f"Mean VRAM Usage (MB): {mean_vram:.2f}")
  
trained_weights = train_sam(dataset, bboxes, weights, output_path, vit="vit_l")
evaluate_model(dataset, trained_weights)


Epoch 1/50 - Loss: 0.1542
Epoch 2/50 - Loss: 0.1022
Epoch 3/50 - Loss: 0.0825
Epoch 4/50 - Loss: 0.0706
Epoch 5/50 - Loss: 0.0693
Epoch 6/50 - Loss: 0.0589
Epoch 7/50 - Loss: 0.0546
Epoch 8/50 - Loss: 0.0522
Epoch 9/50 - Loss: 0.0457
Epoch 10/50 - Loss: 0.0411
Epoch 11/50 - Loss: 0.0394
Epoch 12/50 - Loss: 0.0372
Epoch 13/50 - Loss: 0.0660
Epoch 14/50 - Loss: 0.0533
Epoch 15/50 - Loss: 0.0423
Epoch 16/50 - Loss: 0.0377
Epoch 17/50 - Loss: 0.0345
Epoch 18/50 - Loss: 0.0314
Epoch 19/50 - Loss: 0.0293
Epoch 20/50 - Loss: 0.0289
Epoch 21/50 - Loss: 0.0281
Epoch 22/50 - Loss: 0.0248
Epoch 23/50 - Loss: 0.0238
Epoch 24/50 - Loss: 0.0227
Epoch 25/50 - Loss: 0.0220
Epoch 26/50 - Loss: 0.0217
Epoch 27/50 - Loss: 0.0226
Epoch 28/50 - Loss: 0.0231
Epoch 29/50 - Loss: 0.0229
Epoch 30/50 - Loss: 0.0222
Epoch 31/50 - Loss: 0.0213
Epoch 32/50 - Loss: 0.0218
Epoch 33/50 - Loss: 0.0211
Epoch 34/50 - Loss: 0.0199
Epoch 35/50 - Loss: 0.0201
Epoch 36/50 - Loss: 0.0208
Epoch 37/50 - Loss: 0.0220
Epoch 38/5